# SRCVAE

In [5]:
import os
import glob
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import warnings
import re
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
)

warnings.filterwarnings('ignore')

# ==========================================
# 1. SRCVAE (Proxy Generator Architecture)
# ==========================================
class SRCVAE(nn.Module):
    def __init__(self, x_dim, y_dim=1, z_dim=5, hidden_dim=64):
        """
        x_dim: The number of observable features (dynamic per dataset)
        y_dim: The target dimension (default 1 for binary classification)
        z_dim: The size of the proxy representation to extract
        """
        super(SRCVAE, self).__init__()
        self.x_dim = x_dim
        self.y_dim = y_dim
        self.z_dim = z_dim

        # --- ENCODER: Maps (X, Y) -> Latent Space (mu, logvar) ---
        self.enc_fc1 = nn.Linear(x_dim + y_dim, hidden_dim)
        self.enc_fc2_mu = nn.Linear(hidden_dim, z_dim)
        self.enc_fc2_logvar = nn.Linear(hidden_dim, z_dim)

        # --- DECODER: Maps Latent Space Z -> Reconstructed (X, Y) ---
        self.dec_fc1 = nn.Linear(z_dim, hidden_dim)
        self.dec_fc2_x = nn.Linear(hidden_dim, x_dim)
        self.dec_fc2_y = nn.Linear(hidden_dim, y_dim)

    def encode(self, x, y):
        # Concatenate X and Y as input to the encoder
        h1 = F.relu(self.enc_fc1(torch.cat([x, y], dim=1)))
        return self.enc_fc2_mu(h1), self.enc_fc2_logvar(h1)

    def reparameterize(self, mu, logvar):
        # The reparameterization trick allows gradients to flow backwards through the randomness
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h3 = F.relu(self.dec_fc1(z))
        # Assuming X features are scaled 0-1 or standardized.
        recon_x = self.dec_fc2_x(h3) 
        # Y is binary, so we use sigmoid to reconstruct probabilities
        recon_y = torch.sigmoid(self.dec_fc2_y(h3))
        return recon_x, recon_y

    def forward(self, x, y):
        mu, logvar = self.encode(x, y)
        z = self.reparameterize(mu, logvar)
        recon_x, recon_y = self.decode(z)
        # Returns reconstructed inputs, the proxy (z), and distribution params
        return recon_x, z, recon_y, mu, logvar

# --- VAE LOSS FUNCTION ---
def vae_loss_function(recon_x, x, recon_y, y, mu, logvar):
    # 1. Reconstruction Loss for X (Mean Squared Error)
    MSE_X = F.mse_loss(recon_x, x, reduction='sum')
    # 2. Reconstruction Loss for Y (Binary Cross Entropy)
    BCE_Y = F.binary_cross_entropy(recon_y, y, reduction='sum')
    # 3. KL Divergence (Forces Z into a normal distribution)
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    
    return MSE_X + BCE_Y + KLD


# ==========================================
# 2. ADVERSARIAL FAIRNESS ARCHITECTURES
# ==========================================
class FairPredictor(nn.Module):
    """ The Main Classifier (Predicts Y from X) """
    def __init__(self, x_dim, hidden_dim=64):
        super(FairPredictor, self).__init__()
        self.fc1 = nn.Linear(x_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.out = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        h = F.relu(self.fc1(x))
        h = F.relu(self.fc2(h))
        return torch.sigmoid(self.out(h))

class Adversary(nn.Module):
    """ The Adversary (Predicts the Proxy Z from the Predictor's output) """
    def __init__(self, y_pred_dim=1, z_dim=5, hidden_dim=64):
        super(Adversary, self).__init__()
        self.fc1 = nn.Linear(y_pred_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.out = nn.Linear(hidden_dim, z_dim)

    def forward(self, y_pred):
        h = F.relu(self.fc1(y_pred))
        h = F.relu(self.fc2(h))
        # Z is continuous, so we output linear predictions
        return self.out(h)

In [7]:
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# ==========================================
# STEP 2: SRCVAE TRAINING & PROXY EXTRACTION
# ==========================================

def train_srcvae(model, X_tensor, Y_tensor, epochs=50, batch_size=128, lr=1e-3):
    """
    Trains the SRCVAE to reconstruct X and Y, thereby forcing the 
    latent space to capture the unobserved factors.
    """
    model.train() # Set model to training mode
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    # Create PyTorch DataLoader for efficient batching
    dataset = TensorDataset(X_tensor, Y_tensor)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    for epoch in range(epochs):
        total_loss = 0
        for batch_x, batch_y in dataloader:
            optimizer.zero_grad() # Clear old gradients
            
            # 1. Forward Pass
            recon_x, z, recon_y, mu, logvar = model(batch_x, batch_y)
            
            # 2. Calculate Loss (Reconstruction + KL Divergence)
            loss = vae_loss_function(recon_x, batch_x, recon_y, batch_y, mu, logvar)
            
            # 3. Backpropagation & Optimization
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            
        # Optional: Uncomment to see training progress
        # if (epoch + 1) % 10 == 0:
        #    print(f"    [VAE Training] Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(dataloader):.4f}")
            
    return model

def extract_proxy(model, X_tensor, Y_tensor):
    """
    Passes the data through the trained VAE to extract the latent proxy Z.
    We use the 'mu' (mean) of the distribution as our deterministic proxy.
    """
    model.eval() # Set model to evaluation mode (turns off dropout/randomness)
    with torch.no_grad():
        # Get the mean (mu) of the latent space to use as our stable proxy
        mu, logvar = model.encode(X_tensor, Y_tensor)
        return mu

In [9]:
# ==========================================
# STEP 3: ADVERSARIAL FAIRNESS TRAINING
# ==========================================

def train_fair_classifier(predictor, adversary, X_tensor, Y_tensor, Z_tensor, 
                          epochs=100, batch_size=128, lr_F=1e-3, lr_G=1e-3, lambda_weight=1.0):
    """
    Trains the Predictor (F) and Adversary (G) against each other.
    lambda_weight: Controls how strongly we enforce fairness. 
                   Higher = more fair, but might hurt accuracy.
    """
    predictor.train()
    adversary.train()
    
    # Separate optimizers for the two competing networks
    opt_F = optim.Adam(predictor.parameters(), lr=lr_F)
    opt_G = optim.Adam(adversary.parameters(), lr=lr_G)
    
    # Combine X, Y, and our new Proxy Z into a single dataset
    dataset = TensorDataset(X_tensor, Y_tensor, Z_tensor)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    for epoch in range(epochs):
        for batch_x, batch_y, batch_z in dataloader:
            
            # -------------------------
            # 1. TRAIN THE ADVERSARY (G)
            # -------------------------
            opt_G.zero_grad()
            
            # Get predictor's output (detached so we don't accidentally train the Predictor here)
            y_pred_detached = predictor(batch_x).detach() 
            
            # Adversary tries to guess Z from the predictions
            z_pred = adversary(y_pred_detached)
            
            # Adversary's Goal: Minimize the error between its guess and the real proxy Z
            loss_G = F.mse_loss(z_pred, batch_z)
            
            loss_G.backward()
            opt_G.step()
            
            # -------------------------
            # 2. TRAIN THE PREDICTOR (F)
            # -------------------------
            opt_F.zero_grad()
            
            # Predictor makes a prediction
            y_pred = predictor(batch_x)
            
            # Predictor Goal 1: Be accurate (Minimize Binary Cross Entropy with real Y)
            loss_clf = F.binary_cross_entropy(y_pred, batch_y)
            
            # Predictor Goal 2: Fool the Adversary
            z_pred_live = adversary(y_pred)
            loss_adv = F.mse_loss(z_pred_live, batch_z)
            
            # Total Predictor Loss: Classification Loss MINUS Adversary's Error
            # (By subtracting it, the Predictor wants the Adversary's error to be HUGE)
            loss_F = loss_clf - (lambda_weight * loss_adv)
            
            loss_F.backward()
            opt_F.step()
            
    return predictor

def predict_fairly(predictor, X_tensor):
    """ Generates final debiased predictions. """
    predictor.eval()
    with torch.no_grad():
        probs = predictor(X_tensor)
        preds = (probs >= 0.5).float()
        return probs.numpy(), preds.numpy()

In [12]:
# ==========================================
# STEP 4: PIPELINE INTEGRATION & EVALUATION
# ==========================================
input_folder = r"C:\Users\patri\Documents\Master Semesters\2nd semester\APA\model_testing\SF_and_Hidden_Nodes_Data_Simulation\results\generated_160_csv_10_seeds\full_data"
#r"C:\Users\patri\Documents\Master Semesters\2nd semester\APA\model_testing\FairPFN\synthetic_data"
output_file = "SRCVAE_syn_results.csv"

dataset_files = glob.glob(os.path.join(input_folder, "*.csv"))
print(f"Found {len(dataset_files)} synthetic datasets to process.")

master_results = []

for file_path in dataset_files:
    dataset_name = os.path.basename(file_path)
    print(f"\nProcessing: {dataset_name} with SRCVAE...")
    
    try:
        df = pd.read_csv(file_path)
        
        # --- DYNAMIC FEATURE DISCOVERY ---
        s_cols = [col for col in df.columns if col.startswith('S')]
        x_cols = [col for col in df.columns if col.startswith('X')]
        u_cols = [col for col in df.columns if col.startswith(('U', 'H', 'C'))]
        
        if 'Y' not in df.columns or len(s_cols) == 0:
            print(f"  [SKIPPED] Missing 'Y' or 'S' columns.")
            continue
            
        # Extract features (S_data is ONLY used for final test evaluation)
        s_target = 'S0' if 'S0' in s_cols else s_cols[0]
        S_data = df[[s_target]].values
        X_features = df[x_cols].values
        y_data = df['Y'].values.reshape(-1, 1)
        
        train_size = min(1000, int(len(X_features) * 0.8))
        
        # Split data. Notice S_data is tracked but NOT fed into X_train!
        X_train, X_test, y_train, y_test, S_train, S_test = train_test_split(
            X_features, y_data, S_data, train_size=train_size, random_state=42, stratify=y_data
        )
        
        # Convert to Tensors
        X_train_t = torch.tensor(X_train, dtype=torch.float32)
        y_train_t = torch.tensor(y_train, dtype=torch.float32)
        X_test_t  = torch.tensor(X_test, dtype=torch.float32)
        
        x_dim = X_train.shape[1]
        z_dim = 5 # Latent proxy size (can be tuned)
        
        # --- PHASE 1: VAE PROXY EXTRACTION ---
        vae_model = SRCVAE(x_dim=x_dim, z_dim=z_dim)
        vae_model = train_srcvae(vae_model, X_train_t, y_train_t, epochs=50)
        Z_train_t = extract_proxy(vae_model, X_train_t, y_train_t)
        
        # --- PHASE 2: ADVERSARIAL DEBIASING ---
        predictor = FairPredictor(x_dim=x_dim)
        adversary = Adversary(z_dim=z_dim)
        
        predictor = train_fair_classifier(
            predictor, adversary, X_train_t, y_train_t, Z_train_t, 
            epochs=100, lambda_weight=1.0 # lambda_weight acts as fairness penalty strength
        )
        
        # --- PHASE 3: EVALUATION ---
        prob_preds, predictions = predict_fairly(predictor, X_test_t)
        y_test_flat = y_test.flatten()
        S_test_flat = S_test.flatten()
            
        # 1. Prediction Metrics
        auc = roc_auc_score(y_test_flat, prob_preds)
        acc = accuracy_score(y_test_flat, predictions)
        prec = precision_score(y_test_flat, predictions, zero_division=0)
        rec = recall_score(y_test_flat, predictions, zero_division=0)
        f1 = f1_score(y_test_flat, predictions, zero_division=0)
        
        # 2. Fairness Metrics (Evaluated on true hidden S)
        group_1_mask = (S_test_flat == 1)
        group_0_mask = (S_test_flat == 0)
        
        rate_1 = np.mean(predictions[group_1_mask]) if np.sum(group_1_mask) > 0 else 0
        rate_0 = np.mean(predictions[group_0_mask]) if np.sum(group_0_mask) > 0 else 0
        
        stat_parity_diff = abs(rate_1 - rate_0)
        disp_impact = (rate_1 / rate_0) if rate_0 > 0 else float('inf')
        
        y_true_1, y_pred_1 = y_test_flat[group_1_mask], predictions[group_1_mask]
        tpr_1 = recall_score(y_true_1, y_pred_1, zero_division=0) if len(y_true_1) > 0 else 0
        
        y_true_0, y_pred_0 = y_test_flat[group_0_mask], predictions[group_0_mask]
        tpr_0 = recall_score(y_true_0, y_pred_0, zero_division=0) if len(y_true_0) > 0 else 0
        
        equal_opp_diff = abs(tpr_1 - tpr_0)
        
        # --- RECORD DATA ---
        master_results.append({
            "model_name": "SRCVAE",
            "name_dataset": dataset_name,
            "n_S": len(s_cols),
            "n_X": len(x_cols),
            "n_U": len(u_cols),
            "total_samples": len(df),
            "ROC_AUC": round(auc, 4),
            "Accuracy": round(acc, 4),
            "Precision": round(prec, 4),
            "Recall": round(rec, 4),
            "F1_Score": round(f1, 4),
            "Statistical_Parity_Diff_(ATE)": round(stat_parity_diff, 4),
            "Disparate_Impact_Ratio": round(disp_impact, 4),
            "Equal_Opportunity_Diff": round(equal_opp_diff, 4),
            "Pos_Rate_S1": round(rate_1, 4),
            "Pos_Rate_S0": round(rate_0, 4)
        })
        
        print(f"  [SUCCESS] AUC: {auc:.3f} | ATE: {stat_parity_diff:.3f}")
        
    except Exception as e:
        print(f"  [ERROR] Failed to process. Reason: {str(e)}")

# ==========================================
# 5. SAVE AGGREGATED RESULTS TO CSV
# ==========================================
if len(master_results) > 0:
    results_df = pd.DataFrame(master_results)
    file_exists = os.path.isfile(output_file)
    results_df.to_csv(output_file, mode='a', header=not file_exists, index=False)
    print("\n" + "="*50)
    print(f"ALL JOBS COMPLETE! Processed {len(master_results)} datasets successfully.")
    print(f"Results appended to: {output_file}")
    print("="*50)
else:
    print("\nNo datasets were successfully processed.")

Found 80 synthetic datasets to process.

Processing: SF_Large_Sample_Size_Dataset_Linear_ReLU_50%_10_base_nodes_u_0p0_seed_0_8_nodes_full_data.csv with SRCVAE...
  [SUCCESS] AUC: 0.811 | ATE: 0.095

Processing: SF_Large_Sample_Size_Dataset_Linear_ReLU_50%_10_base_nodes_u_0p0_seed_1_8_nodes_full_data.csv with SRCVAE...
  [SUCCESS] AUC: 0.653 | ATE: 0.016

Processing: SF_Large_Sample_Size_Dataset_Linear_ReLU_50%_10_base_nodes_u_0p0_seed_2_8_nodes_full_data.csv with SRCVAE...
  [SUCCESS] AUC: 0.737 | ATE: 0.208

Processing: SF_Large_Sample_Size_Dataset_Linear_ReLU_50%_10_base_nodes_u_0p0_seed_3_8_nodes_full_data.csv with SRCVAE...
  [SUCCESS] AUC: 0.789 | ATE: 0.081

Processing: SF_Large_Sample_Size_Dataset_Linear_ReLU_50%_10_base_nodes_u_0p0_seed_4_8_nodes_full_data.csv with SRCVAE...
  [SUCCESS] AUC: 0.798 | ATE: 0.264

Processing: SF_Large_Sample_Size_Dataset_Linear_ReLU_50%_10_base_nodes_u_0p0_seed_5_8_nodes_full_data.csv with SRCVAE...
  [SUCCESS] AUC: 0.520 | ATE: 0.162

Processing:

In [11]:
# ==========================================
# STEP 4: SEMI-SYNTHETIC PIPELINE INTEGRATION
# ==========================================
input_folder = r"C:\Users\patri\Documents\Master Semesters\2nd semester\APA\model_testing\FairPFN\semi_synthetic_data"
output_file = "SRCVAE_semi_syn_results.csv"

dataset_files = glob.glob(os.path.join(input_folder, "*.csv"))
print(f"Found {len(dataset_files)} semi-synthetic datasets to process.")

master_results = []

for file_path in dataset_files:
    dataset_name = os.path.basename(file_path)
    print(f"\nProcessing: {dataset_name} with SRCVAE...")
    
    try:
        df = pd.read_csv(file_path)
        
        # --- FILENAME PARAMETER EXTRACTION ---
        name_no_ext = dataset_name.replace('.csv', '')
        bias_level = "HighBias" if "HighBias" in name_no_ext else ("LowBias" if "LowBias" in name_no_ext else "Unknown")
        thresh_match = re.search(r'Thresh([\d\.]+)', name_no_ext)
        threshold = thresh_match.group(1) if thresh_match else "Unknown"
        data_type = "BIASED_ALL" if "BIASED_ALL" in name_no_ext else ("FAIR_ALL" if "FAIR_ALL" in name_no_ext else "Unknown")

        # --- DYNAMIC FEATURE DISCOVERY ---
        y_cols = [col for col in df.columns if col.startswith('Y')]
        if not y_cols:
            print("  [SKIPPED] Missing 'Y' target column.")
            continue
        y_data = df[y_cols[0]].values.reshape(-1, 1)
        
        s_cols = [col for col in df.columns if col.startswith('S') or col.startswith('G')]
        has_protected = len(s_cols) > 0
        
        if has_protected:
            s_target = s_cols[0]
            S_data = df[[s_target]].values
        else:
            print("  [INFO] No protected column found. Evaluating without fairness metrics.")
            S_data = np.zeros((len(df), 1))
            
        x_cols = [col for col in df.columns if col.startswith('X_')]
        X_features = df[x_cols].values
        
        u_cols = [col for col in df.columns if col.startswith(('U_', 'H_', 'C_'))]
        
        train_size = min(1000, int(len(X_features) * 0.8))
        
        # Split data. Notice S_data is tracked but NOT fed into X_train!
        X_train, X_test, y_train, y_test, S_train, S_test = train_test_split(
            X_features, y_data, S_data, train_size=train_size, random_state=42, stratify=y_data
        )
        
        # Convert to Tensors
        X_train_t = torch.tensor(X_train, dtype=torch.float32)
        y_train_t = torch.tensor(y_train, dtype=torch.float32)
        X_test_t  = torch.tensor(X_test, dtype=torch.float32)
        
        x_dim = X_train.shape[1]
        z_dim = 5 # Latent proxy size (can be tuned)
        
        # --- PHASE 1: VAE PROXY EXTRACTION ---
        vae_model = SRCVAE(x_dim=x_dim, z_dim=z_dim)
        vae_model = train_srcvae(vae_model, X_train_t, y_train_t, epochs=50)
        Z_train_t = extract_proxy(vae_model, X_train_t, y_train_t)
        
        # --- PHASE 2: ADVERSARIAL DEBIASING ---
        predictor = FairPredictor(x_dim=x_dim)
        adversary = Adversary(z_dim=z_dim)
        
        predictor = train_fair_classifier(
            predictor, adversary, X_train_t, y_train_t, Z_train_t, 
            epochs=100, lambda_weight=1.0 # lambda_weight acts as fairness penalty strength
        )
        
        # --- PHASE 3: EVALUATION ---
        prob_preds, predictions = predict_fairly(predictor, X_test_t)
        y_test_flat = y_test.flatten()
        S_test_flat = S_test.flatten()
            
        # 1. Prediction Metrics
        auc = roc_auc_score(y_test_flat, prob_preds)
        acc = accuracy_score(y_test_flat, predictions)
        prec = precision_score(y_test_flat, predictions, zero_division=0)
        rec = recall_score(y_test_flat, predictions, zero_division=0)
        f1 = f1_score(y_test_flat, predictions, zero_division=0)
        
        # 2. Fairness Metrics
        if has_protected:
            group_1_mask = (S_test_flat == 1)
            group_0_mask = (S_test_flat == 0)
            
            rate_1 = np.mean(predictions[group_1_mask]) if np.sum(group_1_mask) > 0 else 0
            rate_0 = np.mean(predictions[group_0_mask]) if np.sum(group_0_mask) > 0 else 0
            
            stat_parity_diff = abs(rate_1 - rate_0)
            disp_impact = (rate_1 / rate_0) if rate_0 > 0 else float('inf')
            
            y_true_1, y_pred_1 = y_test_flat[group_1_mask], predictions[group_1_mask]
            tpr_1 = recall_score(y_true_1, y_pred_1, zero_division=0) if len(y_true_1) > 0 else 0
            
            y_true_0, y_pred_0 = y_test_flat[group_0_mask], predictions[group_0_mask]
            tpr_0 = recall_score(y_true_0, y_pred_0, zero_division=0) if len(y_true_0) > 0 else 0
            
            equal_opp_diff = abs(tpr_1 - tpr_0)
        else:
            rate_1 = rate_0 = stat_parity_diff = disp_impact = equal_opp_diff = np.nan
        
        # --- RECORD DATA ---
        master_results.append({
            "model_name": "SRCVAE",
            "name_dataset": dataset_name,
            "bias_level": bias_level,
            "threshold": threshold,
            "data_type": data_type,
            "n_S": len(s_cols),
            "n_X": len(x_cols),
            "n_U": len(u_cols),
            "total_samples": len(df),
            "ROC_AUC": round(auc, 4),
            "Accuracy": round(acc, 4),
            "Precision": round(prec, 4),
            "Recall": round(rec, 4),
            "F1_Score": round(f1, 4),
            "Statistical_Parity_Diff_(ATE)": round(stat_parity_diff, 4) if pd.notna(stat_parity_diff) else np.nan,
            "Disparate_Impact_Ratio": round(disp_impact, 4) if pd.notna(disp_impact) else np.nan,
            "Equal_Opportunity_Diff": round(equal_opp_diff, 4) if pd.notna(equal_opp_diff) else np.nan,
            "Pos_Rate_S1": round(rate_1, 4) if pd.notna(rate_1) else np.nan,
            "Pos_Rate_S0": round(rate_0, 4) if pd.notna(rate_0) else np.nan
        })
        
        if has_protected:
            print(f"  [SUCCESS] AUC: {auc:.3f} | ATE: {stat_parity_diff:.3f}")
        else:
            print(f"  [SUCCESS] AUC: {auc:.3f} | ATE: N/A (No Protected Attribute)")
        
    except Exception as e:
        print(f"  [ERROR] Failed to process. Reason: {str(e)}")

# ==========================================
# 5. SAVE AGGREGATED RESULTS TO CSV
# ==========================================
if len(master_results) > 0:
    results_df = pd.DataFrame(master_results)
    file_exists = os.path.isfile(output_file)
    results_df.to_csv(output_file, mode='a', header=not file_exists, index=False)
    print("\n" + "="*50)
    print(f"ALL JOBS COMPLETE! Processed {len(master_results)} datasets successfully.")
    print(f"Results appended to: {output_file}")
    print("="*50)
else:
    print("\nNo datasets were successfully processed.")

Found 800 semi-synthetic datasets to process.

Processing: HR_N1000_HighBias_Thresh1.0_Seed1000_BIASED_ALL.csv with SRCVAE...
  [SUCCESS] AUC: 0.630 | ATE: 0.010

Processing: HR_N1000_HighBias_Thresh1.0_Seed1000_BIASED_FAIRNESS.csv with SRCVAE...
  [SUCCESS] AUC: 0.410 | ATE: 0.000

Processing: HR_N1000_HighBias_Thresh1.0_Seed1000_BIASED_MASKED.csv with SRCVAE...
  [INFO] No protected column found. Evaluating without fairness metrics.
  [SUCCESS] AUC: 0.683 | ATE: N/A (No Protected Attribute)

Processing: HR_N1000_HighBias_Thresh1.0_Seed1000_FAIR_ALL.csv with SRCVAE...
  [SUCCESS] AUC: 0.493 | ATE: 0.000

Processing: HR_N1000_HighBias_Thresh1.0_Seed100_BIASED_ALL.csv with SRCVAE...
  [SUCCESS] AUC: 0.689 | ATE: 0.000

Processing: HR_N1000_HighBias_Thresh1.0_Seed100_BIASED_FAIRNESS.csv with SRCVAE...
  [SUCCESS] AUC: 0.742 | ATE: 0.028

Processing: HR_N1000_HighBias_Thresh1.0_Seed100_BIASED_MASKED.csv with SRCVAE...
  [INFO] No protected column found. Evaluating without fairness metrics